# Línea realista · un asistente que funcionaría en tiempo real

**Entrega realista: `salidas/submissions/sub_real_adri_noche.csv`** (`real_adri_noche`): mezcla de 5 configuraciones de
LightGBM × 4 semillas, con el snapshot actual y snapshots **pasados** de la sesión. **CV 0.6732 · CV 023 0.6734 · peor
controlador 0.595.**

**Resumen en cuatro líneas**
1. El reto: en cada snapshot (la lista de tareas abiertas del controlador aéreo en un instante) hay que poner arriba la tarea
   en la que está trabajando. Métrica: acierto top-3 (azar ≈ 0.09).
2. La línea realista solo usa lo que un sistema en directo tendría en ese momento: el snapshot actual y los anteriores de
   la sesión. Nada del futuro y ninguna etiqueta de otros controladores.
3. Con solo el snapshot se llega a 0.657–0.664. El **pasado del avión** (cuánto hace que cambió el estado ligado a cada
   tarea y cómo se mueve respecto al borde del sector) sube a **0.670**, y afinar los parámetros y mezclar 5 configuraciones, a
   **0.6732**.
4. Se eligió `real_adri_noche` porque es la mejor en los controladores parecidos al del privado (CV 023), no tiene pesos
   ajustados con el OOF y no depende de una semilla con suerte. Las variantes con mejor CV global bajan en esos controladores.

**Cómo usar este notebook.** Solo necesita la carpeta `entrega/`. **No entrena nada:** lee las salidas ya calculadas de
`entrega/salidas/` y calcula las tablas con `codigo/comun.py`. Las cifras que no se pueden recalcular desde esas salidas
(modelos intermedios cuyos OOF no se guardaron) se citan de `realista/LGB_realista.ipynb` y `realista/noche/INFORME.md` del
repo, y se marcan con **(citado)**.

| Sección | Qué cuenta |
|---|---|
| 1 | Qué es la línea realista y por qué existe |
| 2 | Qué información usa y cuál no |
| 3 | Los cinco bloques de features, desde cero |
| 4 | La cadena de CV: de 0.657 a 0.6732 |
| 5 | La búsqueda de parámetros y la mezcla final |
| 6 | Por qué `real_adri_noche` y no `real_mix` ni `real_noche_sty` |
| 7 | Por qué el segundo paso de sesión no ayuda sin futuro |
| 8 | Qué no funcionó |
| 9 | Comprobación de la entrega y cómo reproducirla |

In [1]:
# Localiza la carpeta entrega/ (subiendo desde el directorio actual) y añade entrega/codigo al path
import sys
from pathlib import Path
import numpy as np, pandas as pd

ENTREGA = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "entrega" / "codigo" / "comun.py").exists()
               or (p / "codigo" / "comun.py").exists())
ENTREGA = ENTREGA if (ENTREGA / "codigo" / "comun.py").exists() else ENTREGA / "entrega"
sys.path.insert(0, str(ENTREGA / "codigo"))
import comun
from comun import labels, hits, medidas, leer, CV023

pd.set_option("display.width", 200, "display.max_columns", 30)
LAB = labels()                                         # sample_id -> target, atco, session, scenario, fold


def oof(nombre):
    o = leer(nombre, "oof")
    assert o is not None, f"falta salidas/oof/{nombre}.csv"
    return o


def tabla(nombres, ref=None):
    # Medidas de cada OOF (y diferencia frente a ref): CV, CV 023, peor controlador, mediana por sesión, escenarios
    r = oof(ref) if ref else None
    filas = []
    for n in nombres:
        m = medidas(oof(n), r if (ref and n != ref) else None)
        f = dict(modelo=n, cv=m["cv"], cv_023=m["cv_023"], peor_controlador=m["peor_controlador"],
                 mediana_sesion=m["mediana_sesion"], esc1=m["escenario_1"], esc2=m["escenario_2"], esc3=m["escenario_3"])
        if "delta" in m:
            f.update(delta_puntos=100 * m["delta"], sesiones_suben_bajan=f"{m['sesiones_suben']}/{m['sesiones_bajan']}",
                     controladores_suben_bajan=f"{m['controladores_suben']}/{m['controladores_bajan']}")
        filas.append(f)
    return pd.DataFrame(filas).set_index("modelo").round(4)


def por_controlador(nombres, ref):
    # Δ (puntos) del acierto por controlador frente a ref; * = los 6 parecidos al 023
    h0 = hits(oof(ref))
    t = {n: (hits(oof(n)).reindex(h0.index) - h0).groupby(LAB.atco.loc[h0.index]).mean() * 100 for n in nombres}
    t = pd.DataFrame(t).T.round(1)
    t.columns = [f"{a:03d}{'*' if a in CV023 else ''}" for a in t.columns]
    return t


print(f"entrega/ en {ENTREGA} · {len(LAB)} snapshots etiquetados · {LAB.atco.nunique()} controladores · "
      f"salidas OOF: {sorted(p.stem for p in (ENTREGA / 'salidas' / 'oof').glob('*.csv') if p.stem.startswith(('real_', 'limpia')))}")

entrega/ en /home/alumno.upv.es/jmflodel/FlowDataLab/entrega · 13175 snapshots etiquetados · 18 controladores · salidas OOF: ['limpia', 'limpia_v1', 'real_adri', 'real_adri_noche', 'real_adri_noche_ses', 'real_mix', 'real_noche_sty']


## 1 · Qué es la línea realista y por qué existe

El organizador permitió al final usar **cualquier información de los datasets**, incluidos los snapshots futuros de la
sesión. Con el futuro, el problema casi se resuelve (CV ~0.92): la tarea en la que trabaja el controlador es la del avión que
**recibe una instrucción en los segundos siguientes**. Eso sirve para la competición, pero no para un asistente real: en
directo, el futuro no existe.

La **línea realista** responde a otra pregunta: *¿cuánto se puede acertar con la información que un sistema tendría en ese
instante?* Es decir, un modelo que se podría poner en una consola de control y que, cada pocos segundos, ordena la lista
de tareas del controlador. El organizador evalúa dos entregas: una con futuro (la de la línea `futuro`) y **esta, la
realista**. Como no hay "mejor de dos" que la cubra, se elige la más robusta para el controlador del privado (el 023).

| Línea | Qué ve en cada snapshot | CV |
|---|---|---|
| limpia (`main`) | solo sus propias filas | 0.664 |
| **realista** | **sus filas + snapshots anteriores de la sesión** | **0.6732** |
| futuro | además, snapshots posteriores y el oráculo (etiquetas de otros controladores en el mismo escenario) | ~0.93 |

## 2 · Qué información usa y cuál no

| Sí | No |
|---|---|
| las filas del snapshot actual (todas las tareas de la lista, de todos los aviones) | ningún snapshot posterior (ni la siguiente instrucción, ni cuándo desaparece la tarea, ni el siguiente cambio de nivel) |
| snapshots **anteriores** de la misma sesión: cómo ha cambiado el estado de cada avión y de cada tarea | la longitud de la racha (`blk_len`) ni el hueco al siguiente snapshot (`gap_next`): miran hacia delante |
| las columnas `a_*`, `x_*`, `k_*` originales (el reto ya las calcula hacia atrás) | etiquetas de otros controladores del mismo escenario (el "oráculo") |
| | el segundo paso de sesión con información futura (sí se probó uno causal: sección 7) |
| | identificadores como features: `sample_id`, `session`, `atco`, `scenario`, `timestamp`, `task_name` |

Dos garantías en el código (`codigo/features_realista.py`):
- `feature_sets()` tiene un `assert` que falla si entre las features aparece una de las columnas que miran al futuro
  (`blk_len`, `gap_next`, `gxc_exitfl_next`); `gxc_exitfl_next` ni siquiera se calcula.
- **Prueba de fuga (citado, `LGB_realista.ipynb` sección 2):** se truncó cada sesión al 40 % y al 70 % y se recalcularon todas
  las features realistas: **0 filas distintas**. Si alguna mirase al futuro, cambiaría al cortar la sesión.

## 3 · Los cinco bloques de features, desde cero

Cada fila es una tarea de la lista (un avión puede tener varias: assume, coordinación, transfer...). El modelo puntúa cada
fila y se quedan las 3 mejores del snapshot. Las features se agrupan en cinco bloques:

| Bloque | Columnas | Qué es | Qué aporta (CV, 3 semillas, **citado**) |
|---|---|---|---|
| `base` | 88 | las columnas originales: tipo de tarea, tiempo y distancia al sector, nivel, si el avión está asumido, antigüedad de la tarea, segundos desde la última instrucción (`a_since_flight`), alertas... | 0.6572 (`real_base`) |
| `logic` | 14 | reglas de un controlador escritas a mano dentro del snapshot: *deadline* según el tipo (cuánto falta para el momento que define la tarea), si la tarea es factible (no se dan instrucciones a un avión que no controlas), si está en la ventana típica de su tipo, orden *earliest-deadline-first* entre las factibles, assume ofrecido y esperando... | +0.0 (`real_blp`, junto con `past`: 0.6571) |
| `past` | 9 | el pasado de la tarea: cuánto lleva en la lista, su rango por antigüedad, si es nueva en la racha, cambios frente al snapshot anterior (`p_d_*`), posición en la racha (`blk_pos`) y hueco con el snapshot anterior (`gap_prev`) | (con `logic`) |
| `causal` | 14 | **segundos desde el último cambio del estado que resuelve cada tarea** (`cz_*`): para una coordinación de entrada, cuándo cambió la coordinación; para un assume, cuándo se ofreció o se asumió; para un transfer, cuándo se asumió; para zona / ruta / conflicto, cuándo cambiaron el rumbo, el nivel autorizado o la alerta. `cz_task` es el que corresponde al tipo de la fila, con su rango en el snapshot, en el tipo y en el avión | **+0.74** (`real_blpc`: 0.6645) |
| `geo` | 26 | geometría del tráfico sin posiciones (los datos no traen lat/lon): velocidad real de acercamiento al borde en los últimos ~30 s y tiempo hasta cruzarlo (`gxa_*`), cuántos otros aviones cruzan el mismo borde a ±60 / ±120 s y a cuántos FL (`gxb_*`), velocidad vertical, FL que faltan hasta el de salida y ritmo necesario (`gxc_*`) | **+0.58** (`real_todo`: 0.6703) |

**La idea de `causal`:** la mayoría de las tareas se trabajan justo después de que *algo cambie* en el avión (llega una
oferta de coordinación, el avión entra, cambia su nivel autorizado). Con futuro se mira cuándo cambiará; sin futuro, lo
más parecido es mirar cuánto hace que cambió. Es el bloque que más aporta.

**La idea de `geo`:** el controlador mira el tráfico, no solo la lista. Un avión que se acerca rápido al borde, o que va a
cruzarlo a la vez que otro a un nivel parecido, pide atención antes de lo que dice la lista.

La celda siguiente construye las features (se leen de `entrega/cache/features_realista.parquet` o se calculan en ~15 s) y
muestra dos hechos sencillos que ayudan a entender los bloques.

In [2]:
from features_realista import construir, BLOQUES
lab, te, fs = construir()
print(f"{lab.sample_id.nunique()} snapshots etiquetados ({len(lab)} filas) · {te.sample_id.nunique()} de test · "
      + ", ".join(f"{b} = {len(fs[b])}" for b in BLOQUES) + f" · total {sum(len(fs[b]) for b in BLOQUES)} columnas")

correcta = lab.target_index == 1
hechos = pd.DataFrame({
    "tarea correcta": [lab.loc[correcta, "cz_task"].notna().mean(), lab.loc[correcta, "logic_feasible"].mean()],
    "resto de tareas": [lab.loc[~correcta, "cz_task"].notna().mean(), lab.loc[~correcta, "logic_feasible"].mean()]},
    index=["causal: el estado que resuelve la tarea ya ha cambiado alguna vez (cz_task no vacío)",
           "logic: la tarea es factible (logic_feasible = 1)"])
hechos.round(3)

13175 snapshots etiquetados (474688 filas) · 1863 de test · base = 88, logic = 14, past = 9, causal = 14, geo = 26 · total 151 columnas


,tarea correcta,resto de tareas
causal: el estado que resuelve la tarea ya ha cambiado alguna vez (cz_task no vacío),0.728,0.525
logic: la tarea es factible (logic_feasible = 1),0.999,0.788


Lectura: en la tarea correcta, el estado que la resuelve ya ha cambiado alguna vez el 73 % de las veces, frente al 53 % en el
resto (`causal`), y casi siempre es factible (`logic`). Pero ninguna regla sola basta: ordenar solo por `logic_edf_rank` acierta en top-3 el ~27 % de las
veces; el modelo combina las 151 columnas.

## 4 · La cadena de CV: de 0.657 a 0.6732

CV = acierto top-3 out-of-fold con `datos/folds.csv` (5 folds agrupados por controlador: el modelo nunca ve en
entrenamiento al controlador que evalúa). *CV 023* = acierto en los 6 controladores de la CV más parecidos al 023 del
privado (002, 011, 010, 006, 017, 015), elegidos por variables sin etiqueta.

| Paso | Modelo | CV | Peor controlador | ¿Se recalcula aquí? |
|---|---|---|---|---|
| solo el snapshot, columnas originales | `real_base` (LightGBM, 3 semillas) | 0.6572 | 0.574 | no (citado) |
| solo el snapshot, con las features de la línea limpia | `limpia` (`lgb_adri_v2` de `main`) | 0.6644 | 0.579 | **sí** |
| + `logic` + `past` + `causal` + `geo` | `real_todo` (parámetros por defecto, 3 semillas) | 0.6703 | 0.596 | no (citado) |
| lo mismo con 5 semillas | `real_adri` | 0.6686 | 0.594 | **sí** |
| parámetros de la búsqueda nocturna, mezcla de 5 × 4 semillas | **`real_adri_noche`** | **0.6732** | 0.595 | **sí** |

`real_todo` y `real_adri` son el mismo modelo con distintas semillas: 0.6703 frente a 0.6686 es **ruido de semilla**, del
mismo tamaño que muchas de las mejoras que se probaron. Por eso nada se adoptó sin confirmarlo con 3 o más semillas.

In [3]:
tabla(["limpia", "real_adri", "real_adri_noche"], ref="real_adri")

,cv,cv_023,peor_controlador,mediana_sesion,esc1,esc2,esc3,delta_puntos,sesiones_suben_bajan,controladores_suben_bajan
modelo,,,,,,,,,,
limpia,0.6644,0.6675,0.5788,0.6783,0.6693,0.6579,0.6663,-0.4175,17/19,8/10
real_adri,0.6686,0.6651,0.5938,0.6818,0.6684,0.6630,0.6752,NaN,NaN,NaN
real_adri_noche,0.6732,0.6734,0.5947,0.6743,0.6752,0.6662,0.6789,0.4630,17/17,10/6


In [4]:
# Δ por controlador (puntos) frente a real_adri (* = los 6 parecidos al 023)
por_controlador(["real_adri_noche"], ref="real_adri")

,001,002*,003,004,005,006*,007,008,009,010*,011*,014,015*,016,017*,018,019,020
real_adri_noche,0.0,1.2,1.9,1.1,1.9,1.3,0.0,-0.4,-2.1,-1.1,1.7,-0.5,-0.9,0.1,2.4,-1.2,0.8,3.3


In [5]:
# ¿En qué tipo de tarea mejora? Acierto top-3 según el tipo de la tarea correcta
tipo = lab.loc[correcta].set_index("sample_id").task_index.map(dict(enumerate(comun.TIPOS)))
t = pd.DataFrame({n: hits(oof(n)) for n in ("limpia", "real_adri", "real_adri_noche")})
t["tipo"] = tipo.reindex(t.index)
g = t.groupby("tipo").agg(muestras=("real_adri", "size"), solo_snapshot=("limpia", "mean"),
                          real_adri=("real_adri", "mean"), real_adri_noche=("real_adri_noche", "mean"))
g["Δ_noche_vs_snapshot"] = g.real_adri_noche - g.solo_snapshot
g.sort_values("muestras", ascending=False).round(3)

,muestras,solo_snapshot,real_adri,real_adri_noche,Δ_noche_vs_snapshot
tipo,,,,,
transfer_aircraft,3527,0.872,0.871,0.872,-0.001
assume_aircraft,3366,0.846,0.840,0.835,-0.011
zone_conflict,2121,0.713,0.717,0.725,0.011
entry_coordination,1068,0.357,0.413,0.434,0.078
return_to_route,937,0.557,0.537,0.555,-0.002
exit_coordination,745,0.079,0.115,0.119,0.040
exit_conditions,448,0.230,0.163,0.185,-0.045
conflict_resolution,390,0.287,0.451,0.431,0.144
qos_improvement,257,0.187,0.140,0.156,-0.031


**Lectura.** Frente a solo el snapshot, el pasado ayuda donde hay un estado cuyo cambio se puede medir: **coordinación de
entrada (+7.8 puntos)**, coordinación de salida (+4.0) y resolución de conflictos (+14.4), justo lo que mide `causal`. Las
tareas fáciles (transfer, assume: ~0.84–0.87) no cambian y algunos tipos pequeños bajan un poco (condiciones de salida −4.5,
qos −3.1). Las tareas difíciles (coordinación de salida 0.12, condiciones de salida 0.19, qos 0.16) siguen sin anticiparse: sin
saber qué hará el controlador después, no dejan huella. La búsqueda de parámetros (`real_adri` → `real_adri_noche`) suma
sobre todo en coordinación de entrada, zona y vuelta a la ruta.

## 5 · La búsqueda de parámetros y la mezcla final

Tras los bloques, lo único que movía la nota eran los **parámetros de LightGBM** (Optuna de Pablo, 1 semilla: +0.6). Adri
dejó de noche una búsqueda (`realista/noche_adri.py` del repo, 2.9 h): mutaciones de las mejores pruebas, 1 semilla cada
una (82 pruebas), y **confirmación de las 6 mejores con 3 semillas**. Cifras **(citado, `realista/noche/INFORME.md`)**:

| Configuración | CV, 1 semilla | **CV, 3 semillas** | Δ vs `real_todo` (3 s., 0.6697) | CV 023 | peor controlador |
|---|---|---|---|---|---|
| `b045` | 0.6721 | **0.6732** | +0.36 | 0.6719 | 0.601 |
| `b030` | 0.6748 | 0.6731 | +0.34 | 0.6675 | 0.594 |
| `b056` | 0.6733 | 0.6721 | +0.24 | 0.6690 | 0.597 |
| `b059` | 0.6728 | 0.6717 | +0.20 | 0.6699 | 0.593 |
| `b060` | 0.6717 | 0.6714 | +0.17 | 0.6673 | 0.594 |
| `b013` (con `extra_trees`) | 0.6718 | 0.6683 | −0.14 | 0.6602 | 0.591 |

- Lo que con 1 semilla parecía +0.5 se quedó en **+0.2 a +0.4** con 3: la mitad era suerte de la semilla.
- Las cinco que confirman se parecen (tabla de abajo): learning rate ~0.02 con 540–1010 árboles, ~37 hojas, `max_bin` 63,
  `reg_lambda` ~21.6 (mucha regularización), subsample 0.94, colsample 0.77 y sin `extra_trees`.
- **Mezcla final:** por cada configuración se promedian 4 semillas; su score se pasa a **z-score dentro del snapshot** (para
  que las cinco estén en la misma escala) y se hace la **media de los cinco con pesos iguales**. Nada se ajusta con el OOF,
  así que la CV de la mezcla no está inflada por haber elegido pesos.
- La mezcla no supera claramente a `b045` solo (0.6742 frente a 0.6732 con 3 semillas por miembro, **citado**): aporta
  **estabilidad** (20 modelos, ninguna configuración con suerte) más que nota.

In [6]:
import entrenar_realista as er
pd.DataFrame(er.CONFIGS).T[["n_estimators", "lr", "leaves", "min_child", "max_bin", "trunc", "reg_lambda", "subsample",
                            "colsample", "min_gain", "cat_smooth", "extra_trees"]].round(4)

,n_estimators,lr,leaves,min_child,max_bin,trunc,reg_lambda,subsample,colsample,min_gain,cat_smooth,extra_trees
b045,756,0.018643,37,25,63,8,21.557216,0.939558,0.772057,0.000501,1.503824,False
b030,1010,0.018643,37,16,63,7,21.557216,0.939558,0.772057,0.000501,1.503824,False
b056,539,0.025639,36,25,63,8,21.557216,0.939558,0.772057,0.000501,1.503824,False
b059,1010,0.023117,37,16,63,5,21.557216,0.939558,0.772057,0.000501,1.503824,False
b060,1010,0.019852,37,16,127,7,21.557216,0.939558,0.772057,0.000501,1.503824,False


## 6 · Por qué `real_adri_noche` y no `real_mix` ni `real_noche_sty`

Después de la búsqueda nocturna se probaron dos variantes con algo más de CV global:
- **`real_mix`** = `real_adri_noche` + un LightGBM `b045` con el bloque **`ctx`** (contexto de la lista: cuántas tareas
  urgentes compiten y lo lejos que está cada una de la más urgente de su tipo). LightGBM puntúa cada fila por separado y no ve
  con qué compite; `ctx` se lo da.
- **`real_noche_sty`** = `real_adri_noche` + 0.25 · un modelo con **`sty`** (estilo del controlador estimado con sus propios
  inputs anteriores: cuánto deja esperar una oferta, a qué distancia acepta, cuándo da la primera orden).

El privado es **un solo controlador nuevo** (el 023). Lo que importa es cómo se comporta el modelo en controladores
parecidos a él, no unas décimas en la media de todos.

In [7]:
tabla(["real_adri_noche", "real_mix", "real_noche_sty"], ref="real_adri_noche")

,cv,cv_023,peor_controlador,mediana_sesion,esc1,esc2,esc3,delta_puntos,sesiones_suben_bajan,controladores_suben_bajan
modelo,,,,,,,,,,
real_adri_noche,0.6732,0.6734,0.5947,0.6743,0.6752,0.6662,0.6789,NaN,NaN,NaN
real_mix,0.6748,0.6670,0.6041,0.6808,0.6766,0.6662,0.6824,0.1518,22/11,10/8
real_noche_sty,0.6745,0.6727,0.6032,0.6776,0.6761,0.6667,0.6814,0.1214,17/11,9/7


In [8]:
# Δ por controlador (puntos) frente a real_adri_noche (* = los 6 parecidos al 023; 020 = el del público)
pc = por_controlador(["real_mix", "real_noche_sty"], ref="real_adri_noche")
pc["parecidos al 023 suben/bajan"] = [f"{(r[[c for c in pc.columns if c.endswith('*')]] > 0).sum()}/"
                                     f"{(r[[c for c in pc.columns if c.endswith('*')]] < 0).sum()}" for _, r in pc.iterrows()]
pc

,001,002*,003,004,005,006*,007,008,009,010*,011*,014,015*,016,017*,018,019,020,parecidos al 023 suben/bajan
real_mix,0.9,-0.3,0.3,-0.2,0.2,-0.5,0.4,1.3,0.4,-1.4,-1.4,0.2,-0.2,0.9,-0.5,1.1,0.9,-0.8,0/6
real_noche_sty,0.0,-0.3,-0.2,-0.9,-0.3,0.3,0.2,0.5,0.3,0.3,-0.2,-0.2,-0.2,0.8,0.1,0.7,0.5,0.0,3/3


**Lectura y decisión.**
- `real_mix` tiene la mejor CV global (0.6748, +0.15) y sube el peor controlador, pero **baja en los 6 controladores
  parecidos al 023** y en el 020: su ganancia viene de controladores que no se parecen a los de test. CV 023: 0.6670 frente a
  0.6734.
- `real_noche_sty` (+0.13 con 4 semillas de `sty`) tampoco sube la CV 023 (0.6727). Con 1 semilla daba +0.22 y un intervalo
  del bootstrap por encima de 0; **con 4 semillas el intervalo es [−0.08, +0.24] y P(mejora) en el privado simulado 0.60
  (citado, sección 5e de `LGB_realista.ipynb`)**: era suerte de la semilla.
- Ninguna variante probada (con `ctx`, `sty`, `trig`, softmax, sus mezclas) sube la CV 023 por encima de 0.6734 (todas
  0.6716–0.6732, **citado**).

**Entrega realista: `real_adri_noche`.** Es la mejor en los controladores parecidos al 023, promedia 20 modelos (no depende de
una semilla) y no tiene ningún peso elegido mirando el OOF. Las diferencias con las alternativas (±0.15) son mucho menores que
el ruido del público de Kaggle (un solo controlador, ±3 puntos), así que tampoco se decide por el público.

## 7 · Por qué el segundo paso de sesión no ayuda sin futuro

En la línea futuro, un **segundo paso de sesión** sube +0.7: un modelo pequeño que combina el score de cada tarea con los de
los snapshots vecinos de la misma racha. Funciona por la estructura de las etiquetas: dentro de una racha el controlador repite
la tarea (94.8 %) y cada tarea se trabaja en una sola racha (98.4 %), así que si después se ve que una tarea "ya se hizo", deja
de ser candidata ("explaining away").

En la realista solo se puede usar la versión **causal** (cada snapshot ve sus scores y los de snapshots *anteriores*:
`realista/feat_session_causal.py` del repo; sin fuga, comprobado). Sin el futuro, pierde casi toda su fuerza: lo que delata la
tarea es lo que ocurre *después*, y lo anterior ya lo recogen `past` y `causal`.

| Modelo | sin segundo paso | con segundo paso causal |
|---|---|---|
| `real_base` (citado) | 0.6572 | 0.6556 |
| `real_todo` (citado) | 0.6703 | 0.6718 |
| `real_adri` (5 semillas) | 0.6686 | 0.6690 (citado) |
| **`real_adri_noche`** | **0.6732** | 0.6722 |

Va de −0.2 a +0.15: ruido. En la entrega **resta** en la CV y en la CV 023, así que se entrega sin él:

In [9]:
tabla(["real_adri_noche", "real_adri_noche_ses"], ref="real_adri_noche")

,cv,cv_023,peor_controlador,mediana_sesion,esc1,esc2,esc3,delta_puntos,sesiones_suben_bajan,controladores_suben_bajan
modelo,,,,,,,,,,
real_adri_noche,0.6732,0.6734,0.5947,0.6743,0.6752,0.6662,0.6789,NaN,NaN,NaN
real_adri_noche_ses,0.6722,0.6719,0.5947,0.6763,0.6739,0.6656,0.6777,-0.1063,17/16,8/8


## 8 · Qué no funcionó

Solo lo necesario para justificar las decisiones (todas las cifras, **citado** de `LGB_realista.ipynb` secciones 2–5e;
diferencias en puntos de CV frente a la referencia de cada prueba, con las mismas semillas):

| Idea | Resultado | Por qué no |
|---|---|---|
| bloque `hist`: tareas resueltas / nuevas respecto al snapshot anterior | −0.6 (3 semillas) | lo que dice ya está en `past` y `causal`; añade ruido |
| mitad hacia atrás del bloque `instr` (instrucciones ya vistas) + atención reciente | −0.55 | con futuro funciona la *siguiente* instrucción, no la anterior |
| peso por tramo (1/√longitud de la racha) | −0.24 | las features de pasado ya corrigen lo que el peso corregía |
| XGBoost `rank:ndcg`, CatBoost YetiRank | −0.5 / −3.0 | LightGBM LambdaRank es mejor; tampoco suman en mezcla |
| stack con los modelos de `main` | −0.3 | la información del snapshot ya está en `base` |
| objetivo softmax sobre la lista, binario, `rank_xendcg` | ≈0 / −0.55 / −1.2 | lambdarank ya optimiza el orden de la lista |
| `extra_trees` y 63 hojas (lo que funciona con futuro) | −0.5 | sin la señal fuerte del futuro, sobreajusta |
| `ctx` (contexto de la lista), `sty` (estilo del controlador) | +0.15 / +0.13 en la CV global | bajan o no suben en los parecidos al 023 (sección 6) |
| más semillas del mismo modelo (1 → 2 en `b045`) | 0.0 | el ruido de semilla se reduce, pero la media no sube |

**Dos lecciones que guían todo lo anterior:**
1. **Con 1 semilla casi todo parece mejorar.** El ruido de semilla (±0.3–0.5) es del tamaño de las mejoras: hay que confirmar
   con 3–4 semillas y mirar sesiones y controladores que suben / bajan, no solo la media.
2. **El techo realista está en ~0.675.** Con futuro, el mismo LightGBM llega a ~0.91: el salto lo da saber qué hace el
   controlador después (siguiente instrucción, cuándo desaparece la tarea), y eso en directo no existe.

## 9 · Comprobación de la entrega y cómo reproducirla

In [10]:
# La submission: 1863 snapshots de test, cada uno una vez, 3 row_index válidos; coincide con el top-3 de preds/
te_rows = pd.read_csv(comun.DATOS / "test.csv", usecols=["sample_id", "row_index"])
for n in ("real_adri_noche", "real_adri_noche_ses", "real_mix", "real_noche_sty", "real_adri", "limpia"):
    s = pd.read_csv(comun.SALIDAS / "submissions" / f"sub_{n}.csv")
    validas = set(map(tuple, te_rows.values))
    filas_ok = all((sid, int(r)) in validas for sid, p in zip(s.sample_id, s.prediction) for r in p.split())
    igual_preds = (comun.to_submission(leer(n, "preds")).set_index("sample_id").prediction
                   .reindex(s.sample_id).values == s.prediction.values).all()
    ok = (set(s.sample_id) == set(te_rows.sample_id) and s.sample_id.is_unique
          and s.prediction.str.split().str.len().eq(3).all() and filas_ok)
    print(f"sub_{n + '.csv':28s} {len(s)} filas · válida: {ok} · igual al top-3 de preds/{n}.csv: {igual_preds}")

sub_real_adri_noche.csv          1863 filas · válida: True · igual al top-3 de preds/real_adri_noche.csv: True
sub_real_adri_noche_ses.csv      1863 filas · válida: True · igual al top-3 de preds/real_adri_noche_ses.csv: True


sub_real_mix.csv                 1863 filas · válida: True · igual al top-3 de preds/real_mix.csv: True
sub_real_noche_sty.csv           1863 filas · válida: True · igual al top-3 de preds/real_noche_sty.csv: True


sub_real_adri.csv                1863 filas · válida: True · igual al top-3 de preds/real_adri.csv: True
sub_limpia.csv                   1863 filas · válida: True · igual al top-3 de preds/limpia.csv: True


**Reproducir la entrega** (desde la raíz del repo o desde `entrega/`; LightGBM con 14 hilos):

```
python entrega/codigo/entrenar_realista.py --rapido     # comprobación: 1 fold, 1 semilla, sin test (~1 min)
python entrega/codigo/entrenar_realista.py              # entrega completa: 5 configuraciones × 4 semillas (~25–40 min)
```

- `codigo/features_realista.py`: construye los bloques `base`, `logic`, `past`, `causal`, `geo` (idénticos celda a celda a
  `realista/features_real.py` del repo) y los guarda en `entrega/cache/features_realista.parquet`.
- `codigo/entrenar_realista.py`: entrena las 5 configuraciones × 4 semillas, imprime las medidas (`comun.medidas`) y guarda
  `salidas/oof/real_adri_noche.csv`, `salidas/preds/real_adri_noche.csv` y `salidas/submissions/sub_real_adri_noche.csv`.
  Cada semilla se guarda en `entrega/cache/realista/`; si se corta, al relanzar continúa.
- Con la semilla 0, el fold 0 de cada configuración da exactamente lo mismo que en la búsqueda nocturna original
  (b045 0.7048, b030 0.7068, b056 0.7024, b059 0.6943, b060 0.7040).

**Salidas que usa este notebook** (`entrega/salidas/`): `real_adri_noche` (entrega), `real_adri_noche_ses` (con segundo
paso), `real_adri` (`real_todo` con los parámetros por defecto, 5 semillas), `real_mix`, `real_noche_sty` (alternativas
descartadas) y `limpia` (la línea limpia, `lgb_adri_v2` en el repo: solo el snapshot).